<a href="https://colab.research.google.com/github/mena-04/FalconFoundry/blob/main/PICO_Softmax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
PICO-Softmax fixed-point golden model: pure base-2 Softmax.

This file models a hardware-friendly datapath:

1. Quantize real input logits to signed fixed-point, default int8 Q3.4.
2. Find max(x).
3. Compute delta_i = x_i - max(x), so each delta_i <= 0.
4. Approximate 2^delta_i using integer right shifts plus a fractional LUT.
5. Accumulate all exponent values.
6. Normalize with a serial shift-subtract fractional divider.

No log2(e) input scaling is applied. The target function is

    y_i = 2^(x_i - max(x)) / sum_j 2^(x_j - max(x)).

The output default is an unsigned 8-bit probability byte interpreted as Q0.8:

    probability ~= output_q / 256.

Because Q0.8 in only 8 bits cannot encode exactly 1.0, the maximum byte 255
represents 255/256.
"""

from __future__ import annotations

import argparse
import csv
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np

In [ ]:
@dataclass(frozen=True)
class FixedSoftmaxConfig:
    vector_len: int = 8
    in_bits: int = 8
    in_frac: int = 4
    exp_frac: int = 14
    out_bits: int = 8
    out_frac: int = 8

    @property
    def in_scale(self) -> int:
        return 1 << self.in_frac

    @property
    def exp_scale(self) -> int:
        return 1 << self.exp_frac

    @property
    def out_scale(self) -> int:
        return 1 << self.out_frac

    @property
    def signed_min(self) -> int:
        return -(1 << (self.in_bits - 1))

    @property
    def signed_max(self) -> int:
        return (1 << (self.in_bits - 1)) - 1

    @property
    def out_max(self) -> int:
        return (1 << self.out_bits) - 1

    @property
    def lut_size(self) -> int:
        return 1 << self.in_frac

    @property
    def lut_width(self) -> int:
        # Q1.exp_frac needs exp_frac + 1 bits because 1.0 is 2^exp_frac.
        return self.exp_frac + 1

In [ ]:
def softmax_base2_float(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Floating-point pure base-2 Softmax, used as the reference."""
    x = np.asarray(x, dtype=np.float64)
    x_shift = x - np.max(x, axis=axis, keepdims=True)
    numerator = np.exp2(x_shift)
    denominator = np.sum(numerator, axis=axis, keepdims=True)
    return numerator / denominator

In [ ]:
def quantize_input(x: np.ndarray, cfg: FixedSoftmaxConfig) -> np.ndarray:
    """Quantize real-valued logits to signed fixed-point integers."""
    q = np.rint(np.asarray(x, dtype=np.float64) * cfg.in_scale).astype(np.int64)
    q = np.clip(q, cfg.signed_min, cfg.signed_max)
    return q

In [ ]:
def dequantize_input(q: np.ndarray, cfg: FixedSoftmaxConfig) -> np.ndarray:
    """Convert signed fixed-point input integers back to real values."""
    return np.asarray(q, dtype=np.float64) / cfg.in_scale

In [ ]:
def make_exp2_negative_lut(cfg: FixedSoftmaxConfig) -> np.ndarray:
    """
    LUT for the fractional part of 2^(-x).

    Address a represents a / 2^in_frac, so the stored value is

        round(2^(-a / 2^in_frac) * 2^exp_frac).

    With in_frac = 4, this is a 16-entry LUT.
    """
    addresses = np.arange(cfg.lut_size, dtype=np.int64)
    frac = addresses / float(cfg.in_scale)
    values = np.exp2(-frac)
    q_values = np.rint(values * cfg.exp_scale).astype(np.int64)
    return q_values

In [ ]:
def exp2_shift_lut_from_delta(
    delta_q: int,
    cfg: FixedSoftmaxConfig,
    lut: np.ndarray,
) -> int:
    """
    Approximate 2^delta for delta <= 0 using a fractional LUT plus shifts.

    delta_q is in signed fixed-point with cfg.in_frac fractional bits.
    Since max subtraction makes delta_q <= 0:

        delta = -magnitude
        magnitude = integer_part + fractional_part
        2^delta = 2^(-integer_part) * 2^(-fractional_part)

    The integer part is implemented as a right shift. The fractional part is
    implemented with a small LUT.
    """
    if delta_q > 0:
        raise ValueError("delta_q must be <= 0 after max subtraction")

    magnitude = -int(delta_q)
    integer_part = magnitude >> cfg.in_frac
    fractional_addr = magnitude & (cfg.in_scale - 1)

    exp_q = int(lut[fractional_addr])
    if integer_part > 0:
        exp_q >>= integer_part
    return exp_q

In [ ]:
def serial_fractional_divide_floor(num: int, den: int, frac_bits: int) -> int:
    """
    Serial shift-subtract divider for a fractional quotient.

    Returns floor((num / den) * 2^frac_bits) for 0 <= num < den.
    For num == den, it returns 2^frac_bits - 1, matching the repeating binary
    fraction 0.111... after frac_bits cycles. This is convenient for an 8-bit
    Q0.8 probability output.
    """
    if den <= 0:
        raise ValueError("denominator must be positive")
    if num < 0:
        raise ValueError("numerator must be non-negative")

    rem = int(num)
    quotient = 0
    for _ in range(frac_bits):
        rem <<= 1
        quotient <<= 1
        if rem >= den:
            rem -= den
            quotient |= 1
    return quotient


In [ ]:
def fixed_base2_softmax_from_quantized(
    x_q: np.ndarray,
    cfg: FixedSoftmaxConfig,
    lut: np.ndarray | None = None,
) -> dict[str, Any]:
    """
    Run the hardware-shaped fixed-point base-2 Softmax model.

    Input x_q must already be signed fixed-point integers.
    """
    x_q = np.asarray(x_q, dtype=np.int64)
    if x_q.ndim != 1:
        raise ValueError("x_q must be a 1D vector")
    if x_q.size != cfg.vector_len:
        raise ValueError(f"expected vector_len={cfg.vector_len}, got {x_q.size}")
    if np.any(x_q < cfg.signed_min) or np.any(x_q > cfg.signed_max):
        raise ValueError("x_q contains values outside the configured signed range")

    if lut is None:
        lut = make_exp2_negative_lut(cfg)

    max_q = int(np.max(x_q))
    delta_q = x_q - max_q

    exp_q = np.array(
        [exp2_shift_lut_from_delta(int(d), cfg, lut) for d in delta_q],
        dtype=np.int64,
    )
    den_q = int(np.sum(exp_q))

    y_q = np.array(
        [serial_fractional_divide_floor(int(n), den_q, cfg.out_frac) for n in exp_q],
        dtype=np.int64,
    )
    y_q = np.clip(y_q, 0, cfg.out_max).astype(np.int64)
    y_real = y_q.astype(np.float64) / cfg.out_scale

    return {
        "x_q": x_q,
        "x_real_quantized": dequantize_input(x_q, cfg),
        "max_q": max_q,
        "delta_q": delta_q,
        "exp_q": exp_q,
        "den_q": den_q,
        "y_q": y_q,
        "y_real": y_real,
        "lut": lut,
    }


In [ ]:
def fixed_base2_softmax(
    x_real: np.ndarray,
    cfg: FixedSoftmaxConfig,
    lut: np.ndarray | None = None,
) -> dict[str, Any]:
    """Quantize real input, then run the fixed-point model."""
    x_q = quantize_input(x_real, cfg)
    return fixed_base2_softmax_from_quantized(x_q, cfg, lut)

In [ ]:
def accuracy_sweep(
    *,
    cfg: FixedSoftmaxConfig,
    samples: int = 10_000,
    seed: int = 3,
    input_std: float = 1.0,
) -> dict[str, Any]:
    """
    Random test of fixed-point model versus floating-point base-2 references.

    Two error numbers are reported:

    * vs_quantized_input: isolates exp/LUT/divider/output quantization error.
    * vs_original_input: includes input quantization error too.
    """
    rng = np.random.default_rng(seed)
    lut = make_exp2_negative_lut(cfg)
    x = rng.normal(loc=0.0, scale=input_std, size=(samples, cfg.vector_len))

    fixed_outputs = np.zeros((samples, cfg.vector_len), dtype=np.float64)
    quantized_float_targets = np.zeros_like(fixed_outputs)
    original_float_targets = softmax_base2_float(x, axis=1)

    output_sum_q = np.zeros(samples, dtype=np.int64)

    for i in range(samples):
        result = fixed_base2_softmax(x[i], cfg, lut)
        fixed_outputs[i] = result["y_real"]
        output_sum_q[i] = int(np.sum(result["y_q"]))
        quantized_float_targets[i] = softmax_base2_float(result["x_real_quantized"])

    err_quantized = np.abs(fixed_outputs - quantized_float_targets)
    err_original = np.abs(fixed_outputs - original_float_targets)

    return {
        "samples": samples,
        "vector_len": cfg.vector_len,
        "input_format": f"signed {cfg.in_bits}-bit Q{cfg.in_bits - cfg.in_frac - 1}.{cfg.in_frac}",
        "exp_format": f"unsigned Q1.{cfg.exp_frac}",
        "output_format": f"unsigned {cfg.out_bits}-bit Q0.{cfg.out_frac}",
        "lut_entries": cfg.lut_size,
        "lut_width_bits": cfg.lut_width,
        "mean_abs_error_vs_quantized_float_base2": float(np.mean(err_quantized)),
        "max_abs_error_vs_quantized_float_base2": float(np.max(err_quantized)),
        "mean_abs_error_vs_original_float_base2": float(np.mean(err_original)),
        "max_abs_error_vs_original_float_base2": float(np.max(err_original)),
        "top1_match_rate_vs_quantized_float_base2": float(
            np.mean(np.argmax(fixed_outputs, axis=1) == np.argmax(quantized_float_targets, axis=1))
        ),
        "mean_output_sum_q": float(np.mean(output_sum_q)),
        "mean_output_sum_real": float(np.mean(output_sum_q) / cfg.out_scale),
        "min_output_sum_q": int(np.min(output_sum_q)),
        "max_output_sum_q": int(np.max(output_sum_q)),
    }


In [ ]:
def write_fixed_vectors_csv(
    path: str | Path,
    *,
    cfg: FixedSoftmaxConfig,
    num_vectors: int = 32,
    seed: int = 4,
    input_std: float = 1.0,
) -> None:
    """Write fixed-point test vectors suitable for a Verilog testbench."""
    path = Path(path)
    rng = np.random.default_rng(seed)
    lut = make_exp2_negative_lut(cfg)
    x = rng.normal(loc=0.0, scale=input_std, size=(num_vectors, cfg.vector_len))

    with path.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "vector_id",
            "index",
            "x_real_original",
            "x_q_signed",
            "x_real_quantized",
            "max_q_signed",
            "delta_q_signed",
            "exp_q_unsigned",
            "den_q_unsigned",
            "y_q_unsigned",
            "y_real_fixed_q0",
            "y_float_base2_from_quantized_input",
        ])

        for vector_id in range(num_vectors):
            result = fixed_base2_softmax(x[vector_id], cfg, lut)
            y_target = softmax_base2_float(result["x_real_quantized"])
            for index in range(cfg.vector_len):
                writer.writerow([
                    vector_id,
                    index,
                    f"{x[vector_id, index]:.12g}",
                    int(result["x_q"][index]),
                    f"{result['x_real_quantized'][index]:.12g}",
                    int(result["max_q"]),
                    int(result["delta_q"][index]),
                    int(result["exp_q"][index]),
                    int(result["den_q"]),
                    int(result["y_q"][index]),
                    f"{result['y_real'][index]:.12g}",
                    f"{y_target[index]:.12g}",
                ])

In [ ]:
def verilog_lut_assignments(cfg: FixedSoftmaxConfig, lut: np.ndarray | None = None) -> str:
    """Return Verilog case items for the fractional exponent LUT."""
    if lut is None:
        lut = make_exp2_negative_lut(cfg)
    addr_width = cfg.in_frac
    data_width = cfg.lut_width
    lines = []
    lines.append("// Fractional LUT for 2^(-frac), pure base-2 Softmax")
    lines.append(f"// addr width = {addr_width}, data width = {data_width}")
    lines.append("always @* begin")
    lines.append("  case (frac_addr)")
    for addr, value in enumerate(lut):
        lines.append(f"    {addr_width}'d{addr}: exp_frac_q = {data_width}'d{int(value)};")
    lines.append(f"    default: exp_frac_q = {data_width}'d{int(lut[0])};")
    lines.append("  endcase")
    lines.append("end")
    return "\n".join(lines)

In [ ]:
def demo_vector(cfg: FixedSoftmaxConfig) -> None:
    """Print one detailed fixed-point example."""
    x = np.array([1.25, -0.50, 0.75, 2.00, -1.00, 0.00, 0.50, -2.00])
    result = fixed_base2_softmax(x, cfg)
    y_float_quantized = softmax_base2_float(result["x_real_quantized"])

    np.set_printoptions(precision=8, suppress=True)
    print("Configuration:")
    print(cfg)
    print("\nInput real:")
    print(x)
    print("\nInput quantized integers:")
    print(result["x_q"])
    print("\nInput after dequantization:")
    print(result["x_real_quantized"])
    print("\nmax_q:")
    print(result["max_q"])
    print("\ndelta_q = x_q - max_q:")
    print(result["delta_q"])
    print("\nexp_q values, Q1.exp_frac scaled integers:")
    print(result["exp_q"])
    print("den_q =", result["den_q"])
    print("\nFixed output y_q, unsigned Q0.out_frac bytes:")
    print(result["y_q"])
    print("sum(y_q) =", int(np.sum(result["y_q"])))
    print("\nFixed output as real y_q / 2^out_frac:")
    print(result["y_real"])
    print("sum =", float(np.sum(result["y_real"])))
    print("\nFloating base-2 reference from quantized input:")
    print(y_float_quantized)
    print("sum =", float(np.sum(y_float_quantized)))
    print("\nError fixed - float_reference:")
    print(result["y_real"] - y_float_quantized)

In [ ]:
def build_config_from_args(args: argparse.Namespace) -> FixedSoftmaxConfig:
    return FixedSoftmaxConfig(
        vector_len=args.vector_len,
        in_bits=args.in_bits,
        in_frac=args.in_frac,
        exp_frac=args.exp_frac,
        out_bits=args.out_bits,
        out_frac=args.out_frac,
    )

In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser(
        description="Fixed-point pure base-2 Softmax golden model."
    )
    parser.add_argument("--vector-len", type=int, default=8)
    parser.add_argument("--in-bits", type=int, default=8)
    parser.add_argument("--in-frac", type=int, default=4)
    parser.add_argument("--exp-frac", type=int, default=14)
    parser.add_argument("--out-bits", type=int, default=8)
    parser.add_argument("--out-frac", type=int, default=8)
    parser.add_argument("--samples", type=int, default=10_000)
    parser.add_argument("--seed", type=int, default=3)
    parser.add_argument("--input-std", type=float, default=1.0)
    parser.add_argument(
        "--csv",
        type=str,
        default="",
        help="Optional CSV path for generated fixed-point test vectors.",
    )
    parser.add_argument("--csv-vectors", type=int, default=32)
    parser.add_argument(
        "--print-lut",
        action="store_true",
        help="Print Verilog case items for the fractional exponent LUT.",
    )
    args = parser.parse_args()

    cfg = build_config_from_args(args)

    demo_vector(cfg)

    print("\nRandom accuracy sweep against floating-point pure base-2 Softmax:")
    metrics = accuracy_sweep(
        cfg=cfg,
        samples=args.samples,
        seed=args.seed,
        input_std=args.input_std,
    )
    for key, value in metrics.items():
        print(f"  {key}: {value}")

    if args.csv:
        write_fixed_vectors_csv(
            args.csv,
            cfg=cfg,
            num_vectors=args.csv_vectors,
            seed=args.seed + 100,
            input_std=args.input_std,
        )
        print(f"\nWrote {args.csv}")

    if args.print_lut:
        print("\nVerilog LUT case items:")
        print(verilog_lut_assignments(cfg))


In [ ]:
import sys

if __name__ == "__main__":
    # Save original sys.argv
    original_argv = sys.argv
    # Temporarily set sys.argv to just the script name
    # This prevents argparse from trying to parse kernel-specific arguments
    sys.argv = [original_argv[0]]
    try:
        main()
    finally:
        # Restore sys.argv to its original state
        sys.argv = original_argv

Configuration:
FixedSoftmaxConfig(vector_len=8, in_bits=8, in_frac=4, exp_frac=14, out_bits=8, out_frac=8)

Input real:
[ 1.25 -0.5   0.75  2.   -1.    0.    0.5  -2.  ]

Input quantized integers:
[ 20  -8  12  32 -16   0   8 -32]

Input after dequantization:
[ 1.25 -0.5   0.75  2.   -1.    0.    0.5  -2.  ]

max_q:
32

delta_q = x_q - max_q:
[-12 -40 -20   0 -48 -32 -24 -64]

exp_q values, Q1.exp_frac scaled integers:
[ 9742  2896  6888 16384  2048  4096  5792  1024]
den_q = 48870

Fixed output y_q, unsigned Q0.out_frac bytes:
[51 15 36 85 10 21 30  5]
sum(y_q) = 253

Fixed output as real y_q / 2^out_frac:
[0.19921875 0.05859375 0.140625   0.33203125 0.0390625  0.08203125
 0.1171875  0.01953125]
sum = 0.98828125

Floating base-2 reference from quantized input:
[0.19933862 0.05926373 0.14095369 0.33524627 0.04190578 0.08381157
 0.11852745 0.02095289]
sum = 0.9999999999999998

Error fixed - float_reference:
[-0.00011987 -0.00066998 -0.00032869 -0.00321502 -0.00284328 -0.00178032
 -0.001

In [ ]:
# Print the LUT as a readable table
cfg = FixedSoftmaxConfig()
lut = make_exp2_negative_lut(cfg)

print("addr | fraction | stored integer | real value | ideal value")
print("-----|----------|----------------|------------|------------")

for addr, value in enumerate(lut):
    fraction = addr / cfg.in_scale
    real_value = value / cfg.exp_scale
    ideal_value = 2 ** (-fraction)
    print(f"{addr:4d} | {fraction:8.4f} | {int(value):14d} | {real_value:10.8f} | {ideal_value:10.8f}")

addr | fraction | stored integer | real value | ideal value
-----|----------|----------------|------------|------------
   0 |   0.0000 |          16384 | 1.00000000 | 1.00000000
   1 |   0.0625 |          15689 | 0.95758057 | 0.95760328
   2 |   0.1250 |          15024 | 0.91699219 | 0.91700404
   3 |   0.1875 |          14387 | 0.87811279 | 0.87812608
   4 |   0.2500 |          13777 | 0.84088135 | 0.84089642
   5 |   0.3125 |          13193 | 0.80523682 | 0.80524517
   6 |   0.3750 |          12634 | 0.77111816 | 0.77110541
   7 |   0.4375 |          12098 | 0.73840332 | 0.73841307
   8 |   0.5000 |          11585 | 0.70709229 | 0.70710678
   9 |   0.5625 |          11094 | 0.67712402 | 0.67712777
  10 |   0.6250 |          10624 | 0.64843750 | 0.64841978
  11 |   0.6875 |          10173 | 0.62091064 | 0.62092891
  12 |   0.7500 |           9742 | 0.59460449 | 0.59460356
  13 |   0.8125 |           9329 | 0.56939697 | 0.56939432
  14 |   0.8750 |           8933 | 0.54522705 | 0.5452

In [ ]:
# To print the Verilog case statement for the LUT
cfg = FixedSoftmaxConfig()
print(verilog_lut_assignments(cfg))

// Fractional LUT for 2^(-frac), pure base-2 Softmax
// addr width = 4, data width = 15
always @* begin
  case (frac_addr)
    4'd0: exp_frac_q = 15'd16384;
    4'd1: exp_frac_q = 15'd15689;
    4'd2: exp_frac_q = 15'd15024;
    4'd3: exp_frac_q = 15'd14387;
    4'd4: exp_frac_q = 15'd13777;
    4'd5: exp_frac_q = 15'd13193;
    4'd6: exp_frac_q = 15'd12634;
    4'd7: exp_frac_q = 15'd12098;
    4'd8: exp_frac_q = 15'd11585;
    4'd9: exp_frac_q = 15'd11094;
    4'd10: exp_frac_q = 15'd10624;
    4'd11: exp_frac_q = 15'd10173;
    4'd12: exp_frac_q = 15'd9742;
    4'd13: exp_frac_q = 15'd9329;
    4'd14: exp_frac_q = 15'd8933;
    4'd15: exp_frac_q = 15'd8555;
    default: exp_frac_q = 15'd16384;
  endcase
end


In [ ]:
# this block for generating csv that contains test vectors, needed for testbench in verilog
cfg = FixedSoftmaxConfig()

write_fixed_vectors_csv(
    "pico_softmax_fixed_vectors.csv",
    cfg=cfg,
    num_vectors=32,
    seed=123,
    input_std=1.0,
)

In [ ]:
import numpy as np
import pandas as pd

cfg = FixedSoftmaxConfig()
lut = make_exp2_negative_lut(cfg)

# Raw signed Q3.4 integer inputs, exactly matching tb_pico_softmax.v
test_vectors_q = {
    "Test 1 - All equal": np.array(
        [16, 16, 16, 16, 16, 16, 16, 16],
        dtype=np.int64,
    ),

    "Test 2 - Increasing": np.array(
        [16, 32, 48, 64, 80, 96, 112, 127],
        dtype=np.int64,
    ),

    "Test 3 - Decreasing": np.array(
        [127, 112, 96, 80, 64, 48, 32, 16],
        dtype=np.int64,
    ),

    "Test 4 - Negative": np.array(
        [-128, -112, -96, -80, -64, -48, -32, -16],
        dtype=np.int64,
    ),

    "Test 5 - Mixed": np.array(
        [32, -16, 112, 0, -64, 80, 16, -32],
        dtype=np.int64,
    ),

    "Test 6 - Dominant": np.array(
        [112, 0, 0, 0, 0, 0, 0, 0],
        dtype=np.int64,
    ),
}

print(f"Loaded {len(test_vectors_q)} directed test vectors.")
python_results = {}

for test_name, x_q in test_vectors_q.items():
    python_results[test_name] = fixed_base2_softmax_from_quantized(
        x_q=x_q,
        cfg=cfg,
        lut=lut,
    )

    result = python_results[test_name]

    print("=" * 70)
    print(test_name)
    print("Input x_q: ", result["x_q"].tolist())
    print("max_q:     ", result["max_q"])
    print("delta_q:   ", result["delta_q"].tolist())
    print("exp_q:     ", result["exp_q"].tolist())
    print("den_q:     ", result["den_q"])
    print("Output y_q:", result["y_q"].tolist())
    print("sum(y_q):  ", int(np.sum(result["y_q"])))

Loaded 6 directed test vectors.
Test 1 - All equal
Input x_q:  [16, 16, 16, 16, 16, 16, 16, 16]
max_q:      16
delta_q:    [0, 0, 0, 0, 0, 0, 0, 0]
exp_q:      [16384, 16384, 16384, 16384, 16384, 16384, 16384, 16384]
den_q:      131072
Output y_q: [32, 32, 32, 32, 32, 32, 32, 32]
sum(y_q):   256
Test 2 - Increasing
Input x_q:  [16, 32, 48, 64, 80, 96, 112, 127]
max_q:      127
delta_q:    [-111, -95, -79, -63, -47, -31, -15, 0]
exp_q:      [133, 267, 534, 1069, 2138, 4277, 8555, 16384]
den_q:      33357
Output y_q: [1, 2, 4, 8, 16, 32, 65, 125]
sum(y_q):   253
Test 3 - Decreasing
Input x_q:  [127, 112, 96, 80, 64, 48, 32, 16]
max_q:      127
delta_q:    [0, -15, -31, -47, -63, -79, -95, -111]
exp_q:      [16384, 8555, 4277, 2138, 1069, 534, 267, 133]
den_q:      33357
Output y_q: [125, 65, 32, 16, 8, 4, 2, 1]
sum(y_q):   253
Test 4 - Negative
Input x_q:  [-128, -112, -96, -80, -64, -48, -32, -16]
max_q:      -16
delta_q:    [-112, -96, -80, -64, -48, -32, -16, 0]
exp_q:      [128, 256,

In [ ]:
rtl_outputs = {
    "Test 1 - All equal": np.array(
        [32, 32, 32, 32, 32, 32, 32, 32],
        dtype=np.int64,
    ),

    "Test 2 - Increasing": np.array(
        [1, 2, 4, 8, 16, 32, 65, 125],
        dtype=np.int64,
    ),

    "Test 3 - Decreasing": np.array(
        [125, 65, 32, 16, 8, 4, 2, 1],
        dtype=np.int64,
    ),

    "Test 4 - Negative": np.array(
        [1, 2, 4, 8, 16, 32, 64, 128],
        dtype=np.int64,
    ),

    "Test 5 - Mixed": np.array(
        [6, 0, 195, 1, 0, 48, 3, 0],
        dtype=np.int64,
    ),

    "Test 6 - Dominant": np.array(
        [242, 1, 1, 1, 1, 1, 1, 1],
        dtype=np.int64,
    ),
}

print("RTL outputs loaded.")
comparison_rows = []

for test_name, result in python_results.items():
    python_y_q = result["y_q"].astype(np.int64)
    rtl_y_q = rtl_outputs[test_name].astype(np.int64)

    error = rtl_y_q - python_y_q
    absolute_error = np.abs(error)

    exact_match = np.array_equal(python_y_q, rtl_y_q)

    comparison_rows.append({
        "Test": test_name,
        "Python output": python_y_q.tolist(),
        "RTL output": rtl_y_q.tolist(),
        "Max error (LSB)": int(np.max(absolute_error)),
        "Mean error (LSB)": float(np.mean(absolute_error)),
        "Status": "PASS" if exact_match else "FAIL",
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

RTL outputs loaded.


,Test,Python output,RTL output,Max error (LSB),Mean error (LSB),Status
0,Test 1 - All equal,"[32, 32, 32, 32, 32, 32, 32, 32]","[32, 32, 32, 32, 32, 32, 32, 32]",0,0.0,PASS
1,Test 2 - Increasing,"[1, 2, 4, 8, 16, 32, 65, 125]","[1, 2, 4, 8, 16, 32, 65, 125]",0,0.0,PASS
2,Test 3 - Decreasing,"[125, 65, 32, 16, 8, 4, 2, 1]","[125, 65, 32, 16, 8, 4, 2, 1]",0,0.0,PASS
3,Test 4 - Negative,"[1, 2, 4, 8, 16, 32, 64, 128]","[1, 2, 4, 8, 16, 32, 64, 128]",0,0.0,PASS
4,Test 5 - Mixed,"[6, 0, 195, 1, 0, 48, 3, 0]","[6, 0, 195, 1, 0, 48, 3, 0]",0,0.0,PASS
5,Test 6 - Dominant,"[242, 1, 1, 1, 1, 1, 1, 1]","[242, 1, 1, 1, 1, 1, 1, 1]",0,0.0,PASS


In [ ]:
passed_tests = int((comparison_df["Status"] == "PASS").sum())
failed_tests = int((comparison_df["Status"] == "FAIL").sum())
overall_max_error = int(comparison_df["Max error (LSB)"].max())

print("=" * 60)
print("PYTHON FIXED-POINT MODEL VS RTL")
print("=" * 60)
print(f"Total tests:       {len(comparison_df)}")
print(f"Passed:            {passed_tests}")
print(f"Failed:            {failed_tests}")
print(f"Maximum RTL error: {overall_max_error} LSB")

if failed_tests == 0:
    print("OVERALL RESULT: PASS")
else:
    print("OVERALL RESULT: FAIL")

PYTHON FIXED-POINT MODEL VS RTL
Total tests:       6
Passed:            6
Failed:            0
Maximum RTL error: 0 LSB
OVERALL RESULT: PASS
